<a href="https://colab.research.google.com/github/dmainagithub/LLMs-Lessons/blob/main/huggingface_text_classification_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Text Classification Tutorial

Note: a GPU is needed in google colab: Runtime -> Change runtime type -> Hardware accelerator -> GPU

### 2. Import necessary commands

In [8]:
# Install dependencies
try:
  import datasets, evaluate, accelerate
  import gradio as gr
except ModuleNotFoundError:
  !pip install -U datasets evaluate accelerate gradio # -U stands for upgrade
  import datasets, evaluate, accelerate
  import gradio as gr

import random

import numpy as np
import pandas as pd

import torch
import transformers

print(f"Using transformers version: {transformers.__version__}")
print(f"Using datasets version: {datasets.__version__}")
print(f"Using torch version: {torch.__version__}")



Using transformers version: 5.16.1
Using datasets version: 4.0.0
Using torch version: 2.11.0+cu128


### 3. Getting a dataset

In [9]:
from datasets import load_dataset

dataset = load_dataset(path="mrdbourke/learn_hf_food_not_food_image_captions")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 250
    })
})

In [10]:
# What features are there
dataset.column_names

{'train': ['text', 'label']}

In [11]:
# Access the training split
dataset["train"]

Dataset({
    features: ['text', 'label'],
    num_rows: 250
})

In [12]:
dataset["train"][0]

{'text': 'Creamy cauliflower curry with garlic naan, featuring tender cauliflower in a rich sauce with cream and spices, served with garlic naan bread.',
 'label': 'food'}

### Inspect random samples

In [18]:
import random

random_indexs = random.sample(range(len(dataset["train"])), 5)
print(random_indexs)

random_samples = dataset["train"][random_indexs]

print(f"[INFO] Random samples from dataset:\n")
for text, label in zip(random_samples["text"], random_samples["label"]):
  print(f" Text: {text} | Label: {label}")


[94, 83, 192, 110, 47]
[INFO] Random samples from dataset:

 Text: Hearty pumpkin curry with toasted pumpkin seeds, featuring sweet pumpkin pieces in a creamy coconut milk sauce, finished with toasted seeds. | Label: food
 Text: Stack of books waiting to be read on a bookshelf | Label: not_food
 Text: Set of spoons stored in a drawer | Label: not_food
 Text: Fragrant vegetable curry with coconut milk and spices, featuring a mix of tender vegetables in a rich, aromatic sauce. | Label: food
 Text: Wooden hanger holding clothes on a rack | Label: not_food


In [19]:
dataset["train"].unique("label")

['food', 'not_food']

In [20]:
# Check the count of each label
from collections import Counter

Counter(dataset["train"]["label"])


Counter({'food': 125, 'not_food': 125})

In [27]:
# Turn our dataset into a dataframe
food_not_food_df = pd.DataFrame(dataset["train"])
food_not_food_df.sample(7)

,text,label
80,Set of curtains draped over a window,not_food
205,Black leather couch adding elegance to a livin...,not_food
147,"Eggplant in a bowl, sprinkled with feta cheese...",food
168,"Rutabaga in a bowl, sprinkled with nutmeg and ...",food
76,Set of bowls stacked on a shelf,not_food
240,Set of glasses gleaming in a cabinet,not_food
133,"Comforting lentil dal curry, featuring nutty l...",food


In [28]:
food_not_food_df["label"].value_counts()

,count
label,
food,125
not_food,125
